# Quick 5T OTA sizing

Give it the headroom (supply, input common mode, output level), a gm/Id and an L for each device, and it returns the widths, the gain, the output and input-common-mode ranges and the VDS margins. It is your own loop (pair, bias source, mirror load, gain = gm / (gds_pair + gds_load), output range) with the VDS values derived instead of typed in.

**How the headroom is derived** (PMOS-input, `kind='p'`; NMOS-input is the mirror image):
- the pair's source sits one VSG above the input common mode, so `vx = vcm + VSG`; the tail then has `VDS = VDD - vx`;
- the pair's VDS on the output side is `vx - vout`; the mirror load's VDS is `vout`; the diode side sits at its own VGS;
- body effect on the pair (bulk at VDD) is included through VSB = `VDD - vx`.

Units: current in uA, W and L in um, voltage in V, gm and gds in uS. PMOS values are magnitudes.

In [ ]:
import math
import pandas as pd
from pygmid import Lookup as lk
from sizing import size_mos
try:
    from IPython.display import display
except Exception:
    display = print

# ---- your tables go here (the same cells as in your other sizing notebooks) ----
nmos = lk("nfet_01v8.mat")
pmos = lk("pfet_01v8.mat")
# def size_mos(mos, ID, gm_id, L, VDS, VSB=0.0): ...   # returns a dict with W, VGS, VDSAT, gm, gds (uS)

In [ ]:
import math
# ---------------- placeholder model (only used when your tables are not defined) ----------------
try:
    nmos, pmos, size_mos
    DEMO = False
except NameError:
    DEMO = True
    UT = 0.0259
    class _Mos:
        def __init__(s, vth0, mucox, n=1.3, gamma=0.5, va=15.0):
            s.vth0, s.mucox, s.n, s.gamma, s.va = vth0, mucox, n, gamma, va
        def vth(s, vsb): return s.vth0 + s.gamma * (math.sqrt(0.8 + max(vsb, 0.0)) - math.sqrt(0.8))
        def idw(s, vgs, vsb, L):                       # uA/um
            vp = (vgs - s.vth(vsb)) / s.n
            i_f = math.log1p(math.exp(vp / (2 * UT))) ** 2
            return i_f * 2 * s.n * s.mucox * UT ** 2 / L
        def lookup(s, out, VGS=None, VDS=None, VSB=0.0, L=1.0, **kw):
            assert out == 'ID_W'
            return s.idw(VGS, VSB, L)
    nmos, pmos = _Mos(0.50, 170.0), _Mos(0.95, 55.0)
    def size_mos(mos, ID, gm_id, L, VDS, VSB=0.0):
        x = 2 / (mos.n * UT * gm_id) - 1
        ic = max((x * x - 1) / 4, 1e-4)
        idw = ic * 2 * mos.n * mos.mucox * UT ** 2 / L
        vgs = mos.vth(VSB) + mos.n * 2 * UT * math.log(math.exp(math.sqrt(ic)) - 1)
        gm = gm_id * ID
        return dict(W=ID / idw, VGS=vgs, VDSAT=2 * UT * (math.sqrt(ic + 0.25) + 0.5), gm=gm, gds=ID / (mos.va * L), gmb=0.2 * gm)
    print('*' * 100)
    print('DEMO MODE: no nmos / pmos / size_mos found, using a placeholder EKV model. The numbers are NOT sky130 numbers.')
    print('*' * 100)

## The sizing function and the report

In [ ]:
import math

def _sz(mos, ID, gm_id, L, VDS, VSB=0.0):
    """size_mos wrapper: returns W, VGS, VDSAT, gm, gds (uS) plus the inputs. PMOS values are magnitudes."""
    r = size_mos(mos, ID=ID, gm_id=gm_id, L=L, VDS=max(float(VDS), 0.05), VSB=max(float(VSB), 0.0))
    nan = float('nan')
    g = lambda *names: next((float(r[n]) for n in names if n in r), nan)
    return dict(W=float(r['W']), VGS=float(r['VGS']), VDSAT=float(r['VDSAT']), gm=g('gm', 'GM'), gds=g('gds', 'GDS'),
                ID=ID, L=L, VDS=float(VDS), gmid=gm_id)

def vgs_fixed_w(mos, ID, W, L, VDS, VSB, fallback=float('nan')):
    """VGS that carries ID in a device of fixed width W (bisection on the ID_W lookup)."""
    target = ID / W
    try:
        lo, hi = 0.05, 1.5
        for _ in range(40):
            mid = 0.5 * (lo + hi)
            if float(mos.lookup('ID_W', VGS=mid, VDS=max(VDS, 0.05), VSB=max(VSB, 0.0), L=L)) < target:
                lo = mid
            else:
                hi = mid
        return 0.5 * (lo + hi)
    except Exception:
        return fallback

def size_ota_5t(nm, pm, vdd, vcm, vout, id_side, gmid, L, kind='p', margin=0.05, vds=None,
                c_load_ff=None, avt=(5.0, 5.0), gamma=0.7, temp=300.0):
    """Five-transistor OTA: input pair, tail, and a current-mirror load.
         kind 'p': PMOS pair and tail, NMOS mirror load (like your ota_p).   kind 'n': NMOS pair and tail, PMOS mirror load.
       Headroom in:  vdd, vcm (input common mode), vout (output DC level), margin (VDS - VDSAT wanted on every device).
       Per device in: gmid = dict(pair, tail, load) and L = dict(pair, tail, load);  id_side = current per branch (uA); tail = 2 * id_side.
       vds (optional) overrides the derived VDS of 'pair', 'tail', 'load', 'load_diode' (V).
       Out: W, gm, gds, VGS, VDSAT per device, gain A, output and input common-mode ranges, VDS - VDSAT margins,
            systematic offset of the mirror (the VDS difference between its two sides), noise and random offset."""
    vds = vds or {}
    pair_m, tail_m, load_m = (pm, pm, nm) if kind == 'p' else (nm, nm, pm)
    vgl = 0.6                                              # diode-connected load: VDS = VGS
    for _ in range(25):
        ld = _sz(load_m, id_side, gmid['load'], L['load'], vds.get('load_diode', vgl))
        if abs(ld['VGS'] - vgl) < 1e-3 or 'load_diode' in vds:
            break
        vgl = ld['VGS']
    v_net1 = ld['VGS'] if kind == 'p' else vdd - ld['VGS']          # voltage of the mirror's diode node
    vgs = 0.7
    for _ in range(30):
        if kind == 'p':
            vx = vcm + vgs; vsb = vdd - vx; vds_pair = vx - vout
        else:
            vx = vcm - vgs; vsb = vx; vds_pair = vout - vx
        pr = _sz(pair_m, id_side, gmid['pair'], L['pair'], vds.get('pair', vds_pair), vsb)
        if abs(pr['VGS'] - vgs) < 1e-3:
            break
        vgs = pr['VGS']
    vds_tail = (vdd - vx) if kind == 'p' else vx
    tl = _sz(tail_m, 2 * id_side, gmid['tail'], L['tail'], vds.get('tail', vds_tail))
    vds_lo = vout if kind == 'p' else vdd - vout
    lo = _sz(load_m, id_side, gmid['load'], L['load'], vds.get('load', vds_lo))
    A = pr['gm'] / (pr['gds'] + lo['gds'])
    # output and input common-mode ranges with every device saturated
    if kind == 'p':
        out_lo, out_hi = lo['VDSAT'] + margin, vx - pr['VDSAT'] - margin
        cm_lo, cm_hi = v_net1 + pr['VDSAT'] + margin - pr['VGS'], vdd - tl['VDSAT'] - margin - pr['VGS']
    else:
        out_lo, out_hi = vx + pr['VDSAT'] + margin, vdd - lo['VDSAT'] - margin
        cm_lo, cm_hi = tl['VDSAT'] + margin + pr['VGS'], v_net1 - pr['VDSAT'] - margin + pr['VGS']
    vds_used = dict(pair=pr['VDS'], tail=tl['VDS'], load=lo['VDS'])
    vdsat = dict(pair=pr['VDSAT'], tail=tl['VDSAT'], load=lo['VDSAT'])
    marg = {k: vds_used[k] - vdsat[k] for k in vds_used}
    # systematic offset: the output-side mirror device sees VDS = vds_lo, the diode side VDS = VGS
    try:
        i_out = ld['W'] * float(load_m.lookup('ID_W', VGS=ld['VGS'], VDS=max(vds_lo, 0.05), VSB=0.0, L=L['load']))
        sys_off_mv = 1e3 * (i_out - id_side) / pr['gm']
    except Exception:
        sys_off_mv = float('nan')
    kT = 1.380649e-23 * temp
    vn2 = 8 * kT * gamma / (pr['gm'] * 1e-6) * (1 + ld['gm'] / pr['gm'])         # V^2/Hz, input referred
    s_pair = avt[0] / math.sqrt(pr['W'] * pr['L'])
    s_load = (ld['gm'] / pr['gm']) * avt[1] / math.sqrt(ld['W'] * ld['L'])
    sig_os = math.sqrt(s_pair ** 2 + s_load ** 2)                                  # input-referred 1 sigma (pair mismatch + mirror mismatch referred to the input)
    res = dict(kind=kind, pair=pr, tail=tl, load=ld, load_out=lo, A=A, A_db=20 * math.log10(A), v_net1=v_net1, vx=vx,
               out_lo=out_lo, out_hi=out_hi, cm_lo=cm_lo, cm_hi=cm_hi, margins=marg, sys_off_mv=sys_off_mv,
               vn_nv=1e9 * math.sqrt(vn2), sigma_os_mv=sig_os, power_uw=vdd * 2 * id_side)
    if c_load_ff:
        res['gbw_mhz'] = pr['gm'] * 1e-6 / (2 * math.pi * c_load_ff * 1e-15) * 1e-6
    res['ok'] = all(m >= margin - 1e-9 for m in marg.values())
    return res

def report_ota(res, title='5T OTA'):
    p, t, l = res['pair'], res['tail'], res['load']
    print(f"{title}  ({'PMOS' if res['kind'] == 'p' else 'NMOS'}-input)")
    print(f"  {'dev':5} {'ID':>6} {'W':>7} {'L':>5} {'gm/Id':>6} {'VGS':>6} {'VDS':>6} {'VDSAT':>6} {'gm':>7} {'gds':>7}")
    for n, d in (('pair', p), ('tail', t), ('load', l)):
        print(f"  {n:5} {d['ID']:6.2f} {d['W']:7.2f} {d['L']:5.2f} {d['gmid']:6.1f} {d['VGS']:6.3f} {d['VDS']:6.3f} {d['VDSAT']:6.3f} {d['gm']:7.1f} {d['gds']:7.3f}")
    print(f"  gain {res['A']:.0f} ({res['A_db']:.1f} dB)" + (f", GBW {res['gbw_mhz']:.1f} MHz" if 'gbw_mhz' in res else ''))
    print(f"  output range {res['out_lo']:.2f} - {res['out_hi']:.2f} V,  input common mode {res['cm_lo']:.2f} - {res['cm_hi']:.2f} V")
    print('  VDS - VDSAT: ' + ', '.join(f'{k} {v:.3f}' for k, v in res['margins'].items()) + ('' if res['ok'] else '   <-- below the margin'))
    print(f"  mirror systematic offset {res['sys_off_mv']:.2f} mV (zero when vout equals the diode node VGS),  thermal noise {res['vn_nv']:.1f} nV/rtHz,  random offset 1 sigma {res['sigma_os_mv']:.2f} mV")

## Inputs

- `KIND`: `'p'` for a PMOS-input OTA (like your summing-stage N-cascode OTA), `'n'` for NMOS-input.
- `VCM`, `VOUT`: the input common mode and the output DC level the OTA has to work at. Choose `VOUT` near the diode-node voltage (the mirror's VGS, printed below) and the mirror's systematic offset goes to zero.
- `ID_SIDE`: current per branch (the tail carries twice this).
- `GMID` and `LSETS`: gm/Id per device, and the L of each device. `LSETS` is a list so you can compare several L choices side by side; add your own dictionaries for different L per device.

In [ ]:
VDD = 1.8
KIND = 'p'                     # 'p' PMOS-input, 'n' NMOS-input
VCM  = 0.5                     # V, input common mode
VOUT = 0.9                     # V, output DC level
ID_SIDE = 5.0                  # uA per branch (tail = 2 * ID_SIDE)
MARGIN = 0.05                  # V: VDS - VDSAT wanted on every device
C_LOAD_FF = 100.0              # fF at the output (for the GBW)
GMID = dict(pair=14, tail=10, load=8)
LSETS = [dict(pair=L, tail=L, load=L) for L in (0.5, 1, 2, 4)]     # or e.g. dict(pair=1, tail=2, load=2)
VDS_OVERRIDE = None            # e.g. dict(pair=0.65, tail=0.35): force a device's VDS instead of deriving it

## Results

In [ ]:
rows, results = [], []
for Ls in LSETS:
    r = size_ota_5t(nmos, pmos, VDD, VCM, VOUT, ID_SIDE, GMID, Ls, kind=KIND, margin=MARGIN, vds=VDS_OVERRIDE, c_load_ff=C_LOAD_FF)
    results.append(r)
    rows.append({'L pair/tail/load': f"{Ls['pair']:g}/{Ls['tail']:g}/{Ls['load']:g}",
                 'W pair': round(r['pair']['W'], 2), 'W tail': round(r['tail']['W'], 2), 'W load': round(r['load']['W'], 2),
                 'gm (uS)': round(r['pair']['gm'], 1), 'gain (dB)': round(r['A_db'], 1), 'GBW (MHz)': round(r.get('gbw_mhz', float('nan')), 1),
                 'out range (V)': f"{r['out_lo']:.2f}-{r['out_hi']:.2f}", 'in CM range (V)': f"{r['cm_lo']:.2f}-{r['cm_hi']:.2f}",
                 'min VDS-VDSAT (V)': round(min(r['margins'].values()), 3), 'sys offset (mV)': round(r['sys_off_mv'], 2),
                 'noise (nV/rtHz)': round(r['vn_nv'], 1), 'sigma offset (mV)': round(r['sigma_os_mv'], 2), 'ok': r['ok']})
df = pd.DataFrame(rows)
display(df)
print(f"diode-node voltage of the mirror (put VOUT near this for zero systematic offset): {results[0]['v_net1']:.3f} V")
if DEMO:
    print('(demo model: not sky130 numbers)')

## Detail for one choice

In [ ]:
PICK = 2                       # index into LSETS
report_ota(results[PICK], title=f'LSETS[{PICK}]')

## Reading it

- **Gain** rises about 6 dB per doubling of L; the price is the area (W grows with L at fixed gm/Id), so the input capacitance and the pole at the output.
- **Input common mode range** is the window of `VCM` over which the pair, tail and mirror stay saturated. If `VCM` is outside it, the report is still printed, but `ok` is False.
- **Output range** is where the output may swing with the pair and the mirror still saturated; it depends on `VCM` through the pair's source voltage.
- **Systematic offset** is the current difference between the two sides of the mirror (diode side at VGS, output side at `VOUT`) divided by gm of the pair.
- Thermal noise only; add flicker from your tables if the bandwidth of interest goes down to the kHz range.